In [1]:
import os
import gc
from glob import glob

import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, 
                          Seq2SeqTrainer, DataCollatorForSeq2Seq)

import warnings
warnings.filterwarnings('ignore')

import wandb

wandb.login()

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: temuujin-razy. Use `wandb login --relogin` to force relogin


In [ ]:
import torch

torch.cuda.empty_cache()

gc.collect()

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run = wandb.init(
    project = 'NUM-Machine-Learning-Lab-3',
    name = "Lab 3 Model Training - Text Summarization T5",
    config = {
        "output_dir": "t5_small_lab3_finetune",
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        'num_train_epochs': 4,
        "train_batch_size": 2,
        "eval_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "fp16": True
    }
)

In [3]:
prefix_val = "summarize"
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

"""
df = pd.DataFrame()

for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)

df['prefix'] = prefix_val

df.to_parquet("arxiv_title_generation.parquet")
"""

df = pd.read_parquet("arxiv_title_generation.parquet")

print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())


Dataframe memory usage
Index               132
target_text     1963896
input_text     17433653
prefix           954492
dtype: int64
Dataframe shape: (14462, 3)

All search terms:
audio+classification
audio+deep+learning
audio+encoding
audio+fast+fourier
audio+fourier
audio+generation
audio+machine+learning
audio+prediction
audio+recognition
audio+representation
audio+restoration
audio+signal
                                         target_text  \
0  Improved Mispronunciation detection system usi...   
1  Improving Factored Hybrid HMM Acoustic Modelin...   
2  Disentangling Style and Speaker Attributes for...   
3  Synthetic speech detection using meta-learning...   
4  A Pre-trained Audio-Visual Transformer for Emo...   

                                          input_text     prefix  
0  This report proposes state-of-the-art research...  summarize  
1  In this work, we show that a factored hybrid h...  summarize  
2  End-to-end neural TTS has shown improved perfo...  summarize  
3  

In [4]:
test_size = 0.2
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

print(arxiv_title_dict)

Training instance count: 11570
Test instance count: 2892

DatasetDict({
    train: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 11570
    })
    test: Dataset({
        features: ['target_text', 'input_text', 'prefix'],
        num_rows: 2892
    })
})


In [5]:
tokenizer_name = "google/t5-efficient-mini"
#tokenizer_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [f"{prefix_val}: " + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = run.config['max_seq_length'],
                             padding = True,
                             truncation = True)

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = run.config["max_seq_length"] // 8,
                       padding = True,
                       truncation = True)
    
    model_inputs["labels"] = labels["input_ids"]
    #model_inputs["decoder_input_ids"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Map:   0%|          | 0/11570 [00:00<?, ? examples/s]

Map:   0%|          | 0/2892 [00:00<?, ? examples/s]

In [6]:
rouge = load_metric("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, 
                                           skip_special_tokens = True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, 
                                            skip_special_tokens = True)

    # Compute ROUGE-1 scores
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]

    # Calculate the mean ROUGE-1 F1 score
    rouge1_f1 = np.mean([score["f"] for score in rouge_scores])

    # Rounds the result to 4 decimal places for cleaner output, and returns it.
    return {"rouge1_f1": round(rouge1_f1, 4)}

In [7]:
model_type = "google/t5-efficient-mini"
#model_type = "t5-small"
#model = T5Model.from_pretrained(model_type)
model = AutoModelForSeq2SeqLM.from_pretrained(model_type)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = model_type)

config = run.config

training_args = Seq2SeqTrainingArguments(
    report_to = "wandb",
    output_dir = config["output_dir"],
    evaluation_strategy = "epoch",
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    per_device_eval_batch_size = config["eval_batch_size"],
    gradient_accumulation_steps = config["gradient_accumulation_steps"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
)

In [8]:
trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_arxiv["train"],
    eval_dataset = tokenized_arxiv["test"],
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

trainer.train()

  0%|          | 0/1928 [00:00<?, ?it/s]

  0%|          | 0/964 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 4.25 GiB. GPU 0 has a total capacity of 11.55 GiB of which 4.30 GiB is free. Including non-PyTorch memory, this process has 6.79 GiB memory in use. Of the allocated memory 5.22 GiB is allocated by PyTorch, and 1.28 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)